# PHAD-Net: Prosodic-Harmonic Aggression Detection Network
## Audio-only Notebook

**Prosodic Dynamics Network (PDN)** — Instead of reducing prosodic features to global statistics (mean pitch, mean energy), we extract frame-level contours and process them with a 1D temporal CNN. This captures *how* prosody evolves — escalation patterns, sudden energy bursts, irregular rhythm — which are cross-lingual signatures of aggressive/hateful speech delivery.

### Architecture
```
  XLS-R-300M ──→ Mean-pool last 4 layers ──→ Attentive pool ──→ Branch 1
  Temporal contours (pitch/energy/centroid/zcr) ──→ PDN (1D CNN) ──→ Branch 2
  Handcrafted features (prosody/VQ/spectral) ──→ MLP ──→ Branch 3
                         ↓
               Gated Fusion + Language Embed → Classifier
```

In [16]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass, field
from collections import defaultdict
import json

import numpy as np
import pandas as pd
import librosa
import torchaudio

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.cuda.amp import GradScaler, autocast

from transformers import Wav2Vec2Model

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import parselmouth
    from parselmouth.praat import call
except ImportError:
    os.system('pip install praat-parselmouth -q')
    import parselmouth
    from parselmouth.praat import call

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

Device: cuda


## Configuration

In [ ]:
@dataclass
class Config:
    base_path: str = "/workspace/ECHO"
    output_path: str = "/workspace/outputs_phad_audio"

    languages: List[str] = field(default_factory=lambda: [
        "bengali", "chinese", "hindi", "english", "french",
        "japanese", "malayalam", "tamil", "telugu", "spanish"
    ])

    test_path: str = "/workspace/ECHO_Test"
    random_seed: int = 42

    sample_rate: int = 16000
    max_duration_sec: float = 10.0
    max_samples: int = 160000

    pretrained_model: str = "facebook/wav2vec2-xls-r-300m"
    encoder_hidden_size: int = 1024
    freeze_encoder_layers: int = 20
    last_k_layers: int = 4 

    hidden_dim: int = 128
    dropout: float = 0.35
    language_embed_dim: int = 32

    pdn_contour_length: int = 100
    pdn_channels: List[int] = field(default_factory=lambda: [32, 64, 64])
    pdn_kernel_sizes: List[int] = field(default_factory=lambda: [7, 5, 3])

    n_mfcc: int = 13

    batch_size: int = 12
    epochs: int = 80
    lr_pretrained: float = 5e-6
    lr_custom: float = 3e-4
    weight_decay: float = 0.03
    warmup_epochs: int = 8
    early_stopping_patience: int = 12
    gradient_accumulation: int = 3
    max_grad_norm: float = 1.0

    focal_gamma: float = 2.0
    label_smoothing: float = 0.05

    use_mixup: bool = True
    mixup_alpha: float = 0.3

    samples_per_language_per_batch: int = 2

    def __post_init__(self):
        self.max_samples = int(self.max_duration_sec * self.sample_rate)
        for d in [self.output_path, f"{self.output_path}/models",
                  f"{self.output_path}/features", f"{self.output_path}/results/figures"]:
            os.makedirs(d, exist_ok=True)

config = Config()
print("=" * 60)
print("PHAD-Net (Audio-Only)")
print("=" * 60)
print(f"  Encoder: {config.pretrained_model}")
print(f"  Last K layers: {config.last_k_layers}")
print(f"  Effective batch: {config.batch_size * config.gradient_accumulation}")

PHAD-Net (Audio-Only)
  Encoder: facebook/wav2vec2-xls-r-300m
  Last K layers: 4
  Effective batch: 36


## Data Preparation

In [18]:
class DataPreparator:
    def __init__(self, cfg: Config):
        self.config = cfg
        self.all_metadata = {}
        self.label_encoder = LabelEncoder()

    @staticmethod
    def normalize_columns(df):
        df.columns = df.columns.str.lower().str.strip()
        mappings = {
            'hate_label': 'label', 'hate_speech': 'label', 'hate': 'label',
            'file_name': 'filename', 'audio_file': 'filename', 'file': 'filename',
        }
        for old, new in mappings.items():
            if old in df.columns and new not in df.columns:
                df = df.rename(columns={old: new})
        if 'gender' not in df.columns:
            df['gender'] = 'unknown'
        return df

    def load_all_metadata(self):
        print("Loading metadata...")
        for lang in tqdm(self.config.languages):
            self.all_metadata[lang] = {}
            lang_path = Path(self.config.base_path) / lang
            for split in ['train', 'validation']:
                split_path = lang_path / split / 'metadata.csv'
                if not split_path.exists():
                    continue
                df = pd.read_csv(split_path)
                df = self.normalize_columns(df)
                df['language'] = lang
                df['split'] = split
                df['audio_path'] = df['filename'].apply(lambda x: str(lang_path / split / x))
                valid = df['audio_path'].apply(lambda p: Path(p).exists())
                df = df[valid]
                self.all_metadata[lang][split] = df
        return self.all_metadata

    def create_splits(self):
        print("\nCreating splits...")
        train_dfs, val_dfs = [], []
        for lang, splits in self.all_metadata.items():
            for sn, df in splits.items():
                if df.empty: continue
                (train_dfs if sn == 'train' else val_dfs).append(df)

        train_df = pd.concat(train_dfs, ignore_index=True)
        val_df = pd.concat(val_dfs, ignore_index=True)

        all_langs = pd.concat([train_df['language'], val_df['language']])
        self.label_encoder.fit(all_langs)
        for df in [train_df, val_df]:
            df['language_id'] = self.label_encoder.transform(df['language'])

        print(f"  Train: {len(train_df)}, Val: {len(val_df)}")
        return train_df, val_df

        train_df = pd.concat(train_dfs, ignore_index=True)
        val_df = pd.concat(val_dfs, ignore_index=True)
        #test_df = pd.concat(test_dfs, ignore_index=True)

        all_langs = pd.concat([train_df['language'], val_df['language']])
        self.label_encoder.fit(all_langs)
        for df in [train_df, val_df, test_df]:
            df['language_id'] = self.label_encoder.transform(df['language'])

        print(f"  Train: {len(train_df)}, Val: {len(val_df)}")
        for split_name, df in [('Train', train_df), ('Val', val_df)]:
            print(f"\n  {split_name} per-language:")
            for lang in self.config.languages:
                sub = df[df['language'] == lang]
                if len(sub) > 0:
                    print(f"    {lang:12s}: {len(sub):4d} ({sub['label'].mean()*100:.0f}% hate)")
        return train_df, val_df

## Feature Extraction

In [19]:
class FeatureExtractor:
    """Extracts global scalar features + temporal contours for PDN."""

    def __init__(self, cfg: Config):
        self.config = cfg

    def extract_prosody(self, audio_path: str) -> Dict[str, float]:
        features = {}
        try:
            sound = parselmouth.Sound(audio_path)
            pitch = call(sound, "To Pitch", 0.0, 75, 600)
            pv = pitch.selected_array['frequency']
            pv = pv[pv > 0]

            if len(pv) > 2:
                features['pitch_mean'] = np.mean(pv)
                features['pitch_std'] = np.std(pv)
                features['pitch_range'] = np.ptp(pv)
                features['pitch_median'] = np.median(pv)
                t = np.arange(len(pv), dtype=np.float64)
                features['pitch_slope'] = np.polyfit(t, pv, 1)[0] if len(t) > 1 else 0.0
                features['pitch_perturbation'] = np.mean(np.abs(np.diff(pv))) / (np.mean(pv) + 1e-8)
            else:
                for k in ['pitch_mean', 'pitch_std', 'pitch_range', 'pitch_median',
                          'pitch_slope', 'pitch_perturbation']:
                    features[k] = 0.0

            intensity = call(sound, "To Intensity", 100, 0.0)
            iv = np.array([call(intensity, "Get value at time", t, "cubic")
                           for t in np.linspace(0, sound.duration, 80)])
            iv = iv[~np.isnan(iv)]
            if len(iv) > 2:
                features['intensity_mean'] = np.mean(iv)
                features['intensity_std'] = np.std(iv)
                features['intensity_range'] = np.ptp(iv)
                features['intensity_dyn_range'] = np.percentile(iv, 95) - np.percentile(iv, 5)
            else:
                for k in ['intensity_mean', 'intensity_std', 'intensity_range', 'intensity_dyn_range']:
                    features[k] = 0.0

            features['voiced_fraction'] = call(pitch, "Count voiced frames") / max(pitch.n_frames, 1)
            pitch_all = pitch.selected_array['frequency']
            vb = (pitch_all > 0).astype(int)
            features['speech_rate_proxy'] = (np.sum(np.abs(np.diff(vb))) // 2 + 1) / max(sound.duration, 0.1)
        except Exception:
            for k in ['pitch_mean', 'pitch_std', 'pitch_range', 'pitch_median',
                      'pitch_slope', 'pitch_perturbation', 'intensity_mean', 'intensity_std',
                      'intensity_range', 'intensity_dyn_range', 'voiced_fraction', 'speech_rate_proxy']:
                features[k] = 0.0
        return features

    def extract_voice_quality(self, audio_path: str) -> Dict[str, float]:
        features = {}
        try:
            sound = parselmouth.Sound(audio_path)
            pp = call(sound, "To PointProcess (periodic, cc)", 75, 600)
            features['jitter'] = call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            features['shimmer'] = call([sound, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
            harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
            features['hnr'] = call(harmonicity, "Get mean", 0, 0)
        except Exception:
            features = {'jitter': 0.0, 'shimmer': 0.0, 'hnr': 0.0}
        return {k: (0.0 if np.isnan(v) else float(v)) for k, v in features.items()}

    def extract_spectral(self, y, sr) -> Dict[str, float]:
        features = {}
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.config.n_mfcc)
        mfcc_d = librosa.feature.delta(mfccs)
        mfcc_dd = librosa.feature.delta(mfccs, order=2)
        for i in range(self.config.n_mfcc):
            features[f'mfcc_{i}_mean'] = float(np.mean(mfccs[i]))
            features[f'mfcc_{i}_std'] = float(np.std(mfccs[i]))
            features[f'mfcc_d_{i}_mean'] = float(np.mean(mfcc_d[i]))
            features[f'mfcc_dd_{i}_mean'] = float(np.mean(mfcc_dd[i]))

        features['spectral_centroid'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        features['spectral_bandwidth'] = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
        features['spectral_rolloff'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
        features['spectral_flatness'] = float(np.mean(librosa.feature.spectral_flatness(y=y)))

        contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
        for i in range(contrast.shape[0]):
            features[f'spectral_contrast_{i}'] = float(np.mean(contrast[i]))

        rms = librosa.feature.rms(y=y)[0]
        features['rms_mean'] = float(np.mean(rms))
        features['rms_std'] = float(np.std(rms))
        features['zcr'] = float(np.mean(librosa.feature.zero_crossing_rate(y)))

        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        features['onset_strength_mean'] = float(np.mean(onset_env))
        features['onset_strength_std'] = float(np.std(onset_env))
        tempo = librosa.beat.tempo(onset_envelope=onset_env, sr=sr)
        features['tempo'] = float(tempo[0]) if len(tempo) > 0 else 0.0
        return features

    def extract_contours(self, y, sr, audio_path) -> np.ndarray:
        L = self.config.pdn_contour_length
        contours = np.zeros((4, L), dtype=np.float32)
        try:
            sound = parselmouth.Sound(audio_path)
            pitch = call(sound, "To Pitch", 0.0, 75, 600)
            pv = pitch.selected_array['frequency']
            if len(pv) > 1:
                voiced = pv > 0
                if voiced.any():
                    pi = np.interp(np.arange(len(pv)), np.where(voiced)[0], pv[voiced])
                else:
                    pi = pv
                contours[0] = np.interp(np.linspace(0, len(pi)-1, L), np.arange(len(pi)), pi)
        except Exception:
            pass
        try:
            hop = int(0.010 * sr)
            rms = librosa.feature.rms(y=y, hop_length=hop)[0]
            if len(rms) > 1:
                contours[1] = np.interp(np.linspace(0, len(rms)-1, L), np.arange(len(rms)), rms)
            cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
            if len(cent) > 1:
                contours[2] = np.interp(np.linspace(0, len(cent)-1, L), np.arange(len(cent)), cent)
            zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
            if len(zcr) > 1:
                contours[3] = np.interp(np.linspace(0, len(zcr)-1, L), np.arange(len(zcr)), zcr)
        except Exception:
            pass
        return contours

    def extract_all(self, audio_path):
        features = {}
        contours = np.zeros((4, self.config.pdn_contour_length), dtype=np.float32)
        try:
            y, sr = librosa.load(audio_path, sr=self.config.sample_rate)
            if len(y) > self.config.max_samples:
                y = y[:self.config.max_samples]
            peak = np.max(np.abs(y))
            if peak > 0:
                y = y / peak
            features.update(self.extract_prosody(audio_path))
            features.update(self.extract_voice_quality(audio_path))
            features.update(self.extract_spectral(y, sr))
            contours = self.extract_contours(y, sr, audio_path)
        except Exception:
            pass
        return features, contours

    def extract_from_dataframe(self, df, name):
        print(f"\nExtracting features for {name} ({len(df)} files)...")
        all_features, all_contours = [], []
        for _, row in tqdm(df.iterrows(), total=len(df)):
            feats, contours = self.extract_all(row['audio_path'])
            feats['filename'] = row['filename']
            feats['label'] = row['label']
            feats['language'] = row['language']
            feats['language_id'] = row['language_id']
            feats['audio_path'] = row['audio_path']
            all_features.append(feats)
            all_contours.append(contours)

        fdf = pd.DataFrame(all_features)
        num_cols = fdf.select_dtypes(include=[np.number]).columns
        fdf[num_cols] = fdf[num_cols].fillna(0)
        contours_arr = np.stack(all_contours, axis=0)
        print(f"  Features: {len(num_cols) - 2} dims, Contours: {contours_arr.shape}")
        return fdf, contours_arr

## Model

In [ ]:
class ProsodicDynamicsNetwork(nn.Module):
    """1D CNN over temporal contours — core novelty."""
    def __init__(self, in_channels=4, channels=[32, 64, 64], kernel_sizes=[7, 5, 3],
                 output_dim=128, dropout=0.3):
        super().__init__()
        layers = []
        prev = in_channels
        for ch, ks in zip(channels, kernel_sizes):
            layers.extend([
                nn.Conv1d(prev, ch, ks, padding=ks // 2),
                nn.BatchNorm1d(ch),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            prev = ch
        self.conv = nn.Sequential(*layers)
        self.attn_pool = nn.Linear(channels[-1], 1)
        self.proj = nn.Sequential(
            nn.Linear(channels[-1], output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, contours):
        x = self.conv(contours).transpose(1, 2)  # (B, L, C)
        w = F.softmax(self.attn_pool(x).squeeze(-1), dim=1).unsqueeze(-1)
        return self.proj((x * w).sum(dim=1))


class GatedFusion(nn.Module):
    def __init__(self, dim, num_branches, dropout=0.3):
        super().__init__()
        total = dim * num_branches
        self.gate = nn.Sequential(
            nn.Linear(total, dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, num_branches), nn.Sigmoid(),
        )
        self.output_proj = nn.Sequential(
            nn.Linear(total, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout),
        )

    def forward(self, branches):
        concat = torch.cat(branches, dim=1)
        gates = self.gate(concat)
        gated = torch.cat([b * gates[:, i:i+1] for i, b in enumerate(branches)], dim=1)
        return self.output_proj(gated)


class PHADNet(nn.Module):
    """
    Three branches:
      1. XLS-R (mean of last K layers) → attentive pool → project
      2. PDN (1D CNN over prosodic contours)
      3. Handcrafted global features → MLP
    Fused via gating, conditioned on language embedding.
    """

    def __init__(self, num_handcrafted_features, num_languages, cfg):
        super().__init__()
        self.config = cfg
        dim = cfg.hidden_dim

        # Branch 1: XLS-R
        print(f"Loading {cfg.pretrained_model}...")
        self.encoder = Wav2Vec2Model.from_pretrained(
            cfg.pretrained_model, output_hidden_states=True
        )
        enc_dim = self.encoder.config.hidden_size

        for p in self.encoder.feature_extractor.parameters():
            p.requires_grad = False
        for p in self.encoder.feature_projection.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.encoder.encoder.layers):
            if i < cfg.freeze_encoder_layers:
                for p in layer.parameters():
                    p.requires_grad = False

        n_layers = len(self.encoder.encoder.layers)
        print(f"  {enc_dim}d, {n_layers} layers, {cfg.freeze_encoder_layers} frozen")

        self.encoder_attn_pool = nn.Sequential(
            nn.Linear(enc_dim, dim), nn.Tanh(), nn.Linear(dim, 1),
        )
        self.encoder_proj = nn.Sequential(
            nn.Linear(enc_dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout),
        )

        # Branch 2: PDN
        self.pdn = ProsodicDynamicsNetwork(
            in_channels=4, channels=cfg.pdn_channels, kernel_sizes=cfg.pdn_kernel_sizes,
            output_dim=dim, dropout=cfg.dropout,
        )

        # Branch 3: Handcrafted
        self.handcrafted_branch = nn.Sequential(
            nn.Linear(num_handcrafted_features, dim * 2),
            nn.LayerNorm(dim * 2), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout),
        )

        self.language_embedding = nn.Embedding(num_languages, cfg.language_embed_dim)

        self.fusion = GatedFusion(dim=dim, num_branches=3, dropout=cfg.dropout)

        clf_in = dim + cfg.language_embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(clf_in, dim), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(dim, dim // 2), nn.GELU(), nn.Dropout(cfg.dropout * 0.5),
            nn.Linear(dim // 2, 2),
        )

    def forward(self, waveform, handcrafted, contours, language_ids):
        # Branch 1: mean of last K layers + attentive pool
        out = self.encoder(waveform, output_hidden_states=True)
        hidden_states = out.hidden_states 
        last_k = hidden_states[-self.config.last_k_layers:]
        audio_seq = torch.stack(last_k, dim=0).mean(dim=0)  

        attn = F.softmax(self.encoder_attn_pool(audio_seq).squeeze(-1), dim=1)
        audio_pooled = (audio_seq * attn.unsqueeze(-1)).sum(dim=1)
        branch1 = self.encoder_proj(audio_pooled)

        # Branch 2: PDN
        branch2 = self.pdn(contours)

        # Branch 3: Handcrafted
        branch3 = self.handcrafted_branch(handcrafted)

        # Fusion
        fused = self.fusion([branch1, branch2, branch3])
        lang_embed = self.language_embedding(language_ids)
        combined = torch.cat([fused, lang_embed], dim=1)
        logits = self.classifier(combined)

        return {'logits': logits, 'fused': fused}

## Dataset & DataLoader

In [21]:
class HateSpeechDataset(Dataset):
    def __init__(self, features_df, contours_array, feature_columns,
                 scaler=None, contour_scaler=None, max_samples=160000, augment=False):
        self.df = features_df.reset_index(drop=True)
        self.max_samples = max_samples
        self.augment = augment

        self.features = self.df[feature_columns].values.astype(np.float32)
        if scaler is not None:
            self.features = scaler.transform(self.features)

        self.contours = contours_array.astype(np.float32)
        if contour_scaler is not None:
            for c in range(4):
                m, s = contour_scaler[c]
                if s > 0:
                    self.contours[:, c, :] = (self.contours[:, c, :] - m) / s

        self.labels = self.df['label'].values.astype(np.int64)
        self.language_ids = self.df['language_id'].values.astype(np.int64)
        self.audio_paths = self.df['audio_path'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.audio_paths[idx])
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        waveform = waveform.squeeze(0)

        if len(waveform) > self.max_samples:
            if self.augment:
                start = np.random.randint(0, len(waveform) - self.max_samples)
                waveform = waveform[start:start + self.max_samples]
            else:
                waveform = waveform[:self.max_samples]
        elif len(waveform) < self.max_samples:
            waveform = F.pad(waveform, (0, self.max_samples - len(waveform)))

        if self.augment:
            if np.random.random() < 0.3:
                waveform = waveform + np.random.uniform(0.001, 0.008) * torch.randn_like(waveform)
            if np.random.random() < 0.3:
                waveform = waveform * np.random.uniform(0.8, 1.2)

        features = torch.tensor(self.features[idx], dtype=torch.float32)
        contours = torch.tensor(self.contours[idx], dtype=torch.float32)
        return waveform, features, contours, self.labels[idx], self.language_ids[idx]


class LanguageBalancedSampler(Sampler):
    def __init__(self, language_ids, labels, samples_per_lang=2, num_batches=None):
        unique_langs = np.unique(language_ids)
        self.num_languages = len(unique_langs)
        self.lang_class_indices = {}
        for lang in unique_langs:
            self.lang_class_indices[lang] = {
                0: np.where((language_ids == lang) & (labels == 0))[0],
                1: np.where((language_ids == lang) & (labels == 1))[0],
            }
        self.samples_per_lang = samples_per_lang
        self.batch_size = samples_per_lang * self.num_languages
        self.num_batches = num_batches or max(1, len(language_ids) // self.batch_size)
        self.total_samples = self.num_batches * self.batch_size

    def __iter__(self):
        indices = []
        for _ in range(self.num_batches):
            for lang in self.lang_class_indices:
                for i in range(self.samples_per_lang):
                    cls = i % 2
                    pool = self.lang_class_indices[lang][cls]
                    if len(pool) == 0:
                        pool = self.lang_class_indices[lang][1 - cls]
                    if len(pool) > 0:
                        indices.append(np.random.choice(pool))
        return iter(indices)

    def __len__(self):
        return self.total_samples


def compute_contour_stats(contours_array):
    return [(float(np.mean(contours_array[:, c])), float(np.std(contours_array[:, c]) + 1e-8))
            for c in range(contours_array.shape[1])]


def create_data_loaders(train_feats, train_cont, val_feats, val_cont, cfg):
    meta = ['filename', 'label', 'language', 'language_id', 'audio_path', 'gender']
    feat_cols = [c for c in train_feats.columns
                 if c not in meta and c not in ['label', 'language_id']
                 and train_feats[c].dtype in [np.float64, np.float32, np.int64]]
    print(f"Handcrafted features: {len(feat_cols)}")

    scaler = StandardScaler()
    scaler.fit(train_feats[feat_cols].values)
    cstats = compute_contour_stats(train_cont)

    kw = dict(feature_columns=feat_cols, scaler=scaler, contour_scaler=cstats, max_samples=cfg.max_samples)
    train_ds = HateSpeechDataset(train_feats, train_cont, augment=True, **kw)
    val_ds = HateSpeechDataset(val_feats, val_cont, augment=False, **kw)
    #test_ds = HateSpeechDataset(test_feats, test_cont, augment=False, **kw)

    sampler = LanguageBalancedSampler(
        train_feats['language_id'].values, train_feats['label'].values,
        samples_per_lang=cfg.samples_per_language_per_batch)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
    #test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                             #num_workers=2, pin_memory=True)
    return train_loader, val_loader, scaler, cstats, feat_cols

## Loss & Training

In [22]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        nc = inputs.size(1)
        with torch.no_grad():
            sm = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
            sm = sm * (1 - self.smoothing) + self.smoothing / nc
        lp = F.log_softmax(inputs, dim=1)
        p = torch.exp(lp)
        loss = -(sm * (1 - p) ** self.gamma * lp).sum(dim=1)
        return loss.mean()


def mixup_data(waveforms, features, contours, labels, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(waveforms.size(0), device=waveforms.device)
    return (lam * waveforms + (1 - lam) * waveforms[idx],
            lam * features + (1 - lam) * features[idx],
            lam * contours + (1 - lam) * contours[idx],
            labels, labels[idx], lam)


class Trainer:
    def __init__(self, model, cfg, device):
        self.model = model
        self.config = cfg
        self.device = device

        pretrained, custom = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if name.startswith('encoder.') and not name.startswith('encoder_'):
                pretrained.append(p)
            else:
                custom.append(p)

        self.optimizer = optim.AdamW([
            {'params': pretrained, 'lr': cfg.lr_pretrained},
            {'params': custom, 'lr': cfg.lr_custom},
        ], weight_decay=cfg.weight_decay)

        self.criterion = FocalLoss(cfg.focal_gamma, cfg.label_smoothing)
        self.scaler = GradScaler()
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=15, T_mult=2, eta_min=1e-6
        )

        self.history = defaultdict(list)
        self.best_f1 = 0
        self.patience = 0

    def warmup_lr(self, epoch):
        if epoch < self.config.warmup_epochs:
            f = (epoch + 1) / self.config.warmup_epochs
            self.optimizer.param_groups[0]['lr'] = self.config.lr_pretrained * f
            self.optimizer.param_groups[1]['lr'] = self.config.lr_custom * f

    def train_epoch(self, loader, epoch):
        self.model.train()
        tot_loss = 0
        all_preds, all_labels = [], []
        self.optimizer.zero_grad()

        pbar = tqdm(loader, desc=f"Train E{epoch+1}")
        for i, (wav, feat, cont, lab, lang) in enumerate(pbar):
            wav = wav.to(self.device)
            feat = feat.to(self.device)
            cont = cont.to(self.device)
            lab = lab.to(self.device)
            lang = lang.to(self.device)

            use_mixup = self.config.use_mixup and np.random.random() < 0.5
            if use_mixup:
                wav, feat, cont, lab_a, lab_b, lam = mixup_data(
                    wav, feat, cont, lab, self.config.mixup_alpha)

            with autocast():
                out = self.model(wav, feat, cont, lang)
                if use_mixup:
                    loss = lam * self.criterion(out['logits'], lab_a) + \
                           (1 - lam) * self.criterion(out['logits'], lab_b)
                else:
                    loss = self.criterion(out['logits'], lab)
                loss = loss / self.config.gradient_accumulation

            self.scaler.scale(loss).backward()

            if (i + 1) % self.config.gradient_accumulation == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()

            tot_loss += loss.item() * self.config.gradient_accumulation
            preds = out['logits'].argmax(1).cpu()
            all_labels.extend((lab_a if use_mixup else lab).cpu().numpy())
            all_preds.extend(preds.numpy())

            if i % 10 == 0:
                pbar.set_postfix(loss=f'{loss.item() * self.config.gradient_accumulation:.4f}')

        n = max(len(loader), 1)
        return {
            'loss': tot_loss / n,
            'f1': f1_score(all_labels, all_preds, zero_division=0),
            'acc': accuracy_score(all_labels, all_preds),
        }

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        tot_loss = 0
        all_preds, all_labels, all_probs = [], [], []

        for wav, feat, cont, lab, lang in loader:
            wav, feat, cont = wav.to(self.device), feat.to(self.device), cont.to(self.device)
            lab, lang = lab.to(self.device), lang.to(self.device)

            out = self.model(wav, feat, cont, lang)
            loss = self.criterion(out['logits'], lab)
            tot_loss += loss.item()

            probs = F.softmax(out['logits'], dim=1)[:, 1].cpu()
            all_probs.extend(probs.numpy())
            all_preds.extend(out['logits'].argmax(1).cpu().numpy())
            all_labels.extend(lab.cpu().numpy())

        try:
            auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            auc = 0.5

        return {
            'loss': tot_loss / max(len(loader), 1),
            'f1': f1_score(all_labels, all_preds, zero_division=0),
            'precision': precision_score(all_labels, all_preds, zero_division=0),
            'recall': recall_score(all_labels, all_preds, zero_division=0),
            'acc': accuracy_score(all_labels, all_preds),
            'auc': auc,
        }

    def train(self, train_loader, val_loader):
        print(f"\n{'='*60}\nTRAINING PHAD-Net (Audio-Only)\n{'='*60}")

        for epoch in range(self.config.epochs):
            self.warmup_lr(epoch)
            tm = self.train_epoch(train_loader, epoch)
            vm = self.validate(val_loader)

            if epoch >= self.config.warmup_epochs:
                self.scheduler.step(epoch - self.config.warmup_epochs)

            gap = tm['f1'] - vm['f1']
            lr = self.optimizer.param_groups[1]['lr']

            print(f"\nEpoch {epoch+1}/{self.config.epochs} | LR: {lr:.2e}")
            print(f"  Train — Loss: {tm['loss']:.4f}, F1: {tm['f1']:.4f}, Acc: {tm['acc']:.4f}")
            print(f"  Val   — Loss: {vm['loss']:.4f}, F1: {vm['f1']:.4f}, Acc: {vm['acc']:.4f}, P: {vm['precision']:.3f}, R: {vm['recall']:.3f}, AUC: {vm['auc']:.4f}")
            print(f"  Gap: {gap:.4f}")

            for k, v in tm.items(): self.history[f'train_{k}'].append(v)
            for k, v in vm.items(): self.history[f'val_{k}'].append(v)

            if vm['f1'] > self.best_f1:
                self.best_f1 = vm['f1']
                self.patience = 0
                torch.save({
                    'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                    'val_f1': self.best_f1, 'val_auc': vm['auc'],
                }, f"{self.config.output_path}/models/best_model.pt")
                print(f"  ✓ New best (F1: {self.best_f1:.4f})")
            else:
                self.patience += 1

            if self.patience >= self.config.early_stopping_patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
            torch.cuda.empty_cache()

        ckpt = f"{self.config.output_path}/models/best_model.pt"
        if os.path.exists(ckpt):
            state = torch.load(ckpt, map_location=self.device)
            self.model.load_state_dict(state['model_state_dict'])
            print(f"\nLoaded best model from epoch {state['epoch']+1}")
        return self.model, dict(self.history)

## Evaluation

In [ ]:
class Evaluator:
    def __init__(self, cfg, device):
        self.config = cfg
        self.device = device

    @staticmethod
    def find_optimal_threshold(labels, probs):
        best_s, best_t = 0, 0.5
        for t in np.arange(0.25, 0.75, 0.02):
            s = f1_score(labels, (probs >= t).astype(int), zero_division=0)
            if s > best_s: best_s, best_t = s, t
        return best_t, best_s

    @torch.no_grad()
    def evaluate(self, model, test_loader, test_df):
        model.eval()
        all_probs, all_labels = [], []

        for wav, feat, cont, lab, lang in tqdm(test_loader, desc="Evaluating"):
            wav, feat, cont = wav.to(self.device), feat.to(self.device), cont.to(self.device)
            lang = lang.to(self.device)
            out = model(wav, feat, cont, lang)
            probs = F.softmax(out['logits'], dim=1)[:, 1].cpu()
            all_probs.extend(probs.numpy())
            all_labels.extend(lab.numpy())

        all_probs = np.array(all_probs)
        all_labels = np.array(all_labels)
        languages = test_df['language'].values[:len(all_labels)]

        global_thresh, _ = self.find_optimal_threshold(all_labels, all_probs)
        per_lang_thresholds = {}
        for lang in np.unique(languages):
            m = languages == lang
            if m.sum() > 10:
                per_lang_thresholds[lang], _ = self.find_optimal_threshold(all_labels[m], all_probs[m])
            else:
                per_lang_thresholds[lang] = global_thresh

        all_preds = np.array([int(p >= per_lang_thresholds.get(l, global_thresh))
                              for l, p in zip(languages, all_probs)])

        results = {
            'threshold': global_thresh,
            'accuracy': accuracy_score(all_labels, all_preds),
            'precision': precision_score(all_labels, all_preds, zero_division=0),
            'recall': recall_score(all_labels, all_preds, zero_division=0),
            'f1': f1_score(all_labels, all_preds, zero_division=0),
            'auc_roc': roc_auc_score(all_labels, all_probs),
            'auc_pr': average_precision_score(all_labels, all_probs),
        }

        results['per_language'] = {}
        for lang in np.unique(languages):
            m = languages == lang
            if m.sum() > 0:
                t = per_lang_thresholds.get(lang, global_thresh)
                p = (all_probs[m] >= t).astype(int)
                results['per_language'][lang] = {
                    'n': int(m.sum()), 'threshold': t,
                    'f1': f1_score(all_labels[m], p, zero_division=0),
                    'precision': precision_score(all_labels[m], p, zero_division=0),
                    'recall': recall_score(all_labels[m], p, zero_division=0),
                }

        print(f"\nProb distribution:")
        print(f"  Non-hate: {np.percentile(all_probs[all_labels==0], [10,25,50,75,90])}")
        print(f"  Hate:     {np.percentile(all_probs[all_labels==1], [10,25,50,75,90])}")

        self._plot_results(all_labels, all_probs, all_preds, results, languages)
        return results

    def _plot_results(self, labels, probs, preds, results, languages):
        fig_dir = f"{self.config.output_path}/results/figures"
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        cm = confusion_matrix(labels, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
                    xticklabels=['Non-Hate', 'Hate'], yticklabels=['Non-Hate', 'Hate'])
        axes[0, 0].set_title('Confusion Matrix')

        fpr, tpr, _ = roc_curve(labels, probs)
        axes[0, 1].plot(fpr, tpr, label=f'AUC={results["auc_roc"]:.3f}')
        axes[0, 1].plot([0,1],[0,1],'k--')
        axes[0, 1].set_title('ROC'); axes[0, 1].legend()

        axes[0, 2].hist(probs[labels==0], bins=50, alpha=0.6, label='Non-hate', density=True)
        axes[0, 2].hist(probs[labels==1], bins=50, alpha=0.6, label='Hate', density=True)
        axes[0, 2].set_title('Probability Distribution'); axes[0, 2].legend()

        langs = sorted(results['per_language'].keys())
        f1s = [results['per_language'][l]['f1'] for l in langs]
        colors = ['#2ecc71' if f > 0.7 else '#f39c12' if f > 0.5 else '#e74c3c' for f in f1s]
        axes[1, 0].bar(langs, f1s, color=colors)
        axes[1, 0].axhline(results['f1'], color='blue', linestyle='--', label=f'Overall: {results["f1"]:.3f}')
        axes[1, 0].set_title('Per-Language F1'); axes[1, 0].legend()
        axes[1, 0].set_xticklabels(langs, rotation=45, ha='right'); axes[1, 0].set_ylim(0, 1)

        precs = [results['per_language'][l]['precision'] for l in langs]
        recs = [results['per_language'][l]['recall'] for l in langs]
        x = np.arange(len(langs)); w = 0.35
        axes[1, 1].bar(x-w/2, precs, w, label='P', alpha=0.8)
        axes[1, 1].bar(x+w/2, recs, w, label='R', alpha=0.8)
        axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(langs, rotation=45, ha='right')
        axes[1, 1].set_title('P vs R'); axes[1, 1].legend(); axes[1, 1].set_ylim(0, 1)

        thresholds = [results['per_language'][l]['threshold'] for l in langs]
        axes[1, 2].bar(langs, thresholds, color='purple', alpha=0.7)
        axes[1, 2].axhline(0.5, color='red', linestyle='--', label='Default')
        axes[1, 2].set_title('Optimal Thresholds'); axes[1, 2].legend()
        axes[1, 2].set_xticklabels(langs, rotation=45, ha='right')

        plt.tight_layout()
        plt.savefig(f"{fig_dir}/evaluation.png", dpi=150, bbox_inches='tight')
        plt.show()

## Pipeline

In [24]:
def compute_val_thresholds(model, val_loader, val_df, device):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for wav, feat, cont, lab, lang in val_loader:
            wav, feat, cont = wav.to(device), feat.to(device), cont.to(device)
            lang = lang.to(device)
            out = model(wav, feat, cont, lang)
            probs = F.softmax(out['logits'], dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(lab.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    languages = val_df['language'].values[:len(all_labels)]

    thresholds = {}
    for lang in np.unique(languages):
        m = languages == lang
        if m.sum() > 5:
            best_t, best_f1 = 0.5, 0
            for t in np.arange(0.25, 0.75, 0.02):
                f = f1_score(all_labels[m], (all_probs[m] >= t).astype(int), zero_division=0)
                if f > best_f1:
                    best_f1, best_t = f, t
            thresholds[lang] = best_t
            print(f"  {lang:12s}: τ={best_t:.2f} (F1={best_f1:.3f})")
        else:
            thresholds[lang] = 0.5

    return thresholds

In [ ]:
def run_inference(model, config, label_encoder, scaler, contour_stats, feat_cols, thresholds):
    model.eval()
    test_path = Path(config.test_path)
    extractor = FeatureExtractor(config)
    print(f"\n{'='*60}\nINFERENCE on {test_path}\n{'='*60}")

    for lang in config.languages:
        meta_path = test_path / lang / 'metadata.csv'
        if not meta_path.exists():
            print(f"  {lang}: skipping"); continue

        df = pd.read_csv(meta_path)
        df.columns = df.columns.str.lower().str.strip()
    
        audio_col = None
        for c in ['audio', 'filename', 'file_name', 'file', 'audio_file', 'path']:
            if c in df.columns: audio_col = c; break
        if audio_col is None:
            print(f"  {lang}: no audio column found in {list(df.columns)}, skipping"); continue
        df['audio_path'] = df[audio_col].apply(lambda x: str(test_path / lang / x))

        lang_id = label_encoder.transform([lang])[0]

        all_features, all_contours = [], []
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  {lang}"):
            feats, cont = extractor.extract_all(row['audio_path'])
            all_features.append(feats)
            all_contours.append(cont)

        feat_df = pd.DataFrame(all_features)

        for c in feat_cols:
            if c not in feat_df.columns:
                feat_df[c] = 0.0
        feat_vals = scaler.transform(feat_df[feat_cols].fillna(0).values).astype(np.float32)

        contours_arr = np.stack(all_contours, axis=0).astype(np.float32)
        for c in range(4):
            m, s = contour_stats[c]
            if s > 0: contours_arr[:, c, :] = (contours_arr[:, c, :] - m) / s

        all_probs = []
        bs = config.batch_size
        for start in range(0, len(df), bs):
            end = min(start + bs, len(df))

            wavs = []
            for idx in range(start, end):
                wav, sr = torchaudio.load(df['audio_path'].values[idx])
                if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
                if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
                wav = wav.squeeze(0)
                if len(wav) > config.max_samples: wav = wav[:config.max_samples]
                elif len(wav) < config.max_samples: wav = F.pad(wav, (0, config.max_samples - len(wav)))
                wavs.append(wav)

            wav_batch = torch.stack(wavs).to(DEVICE)
            feat_batch = torch.tensor(feat_vals[start:end], dtype=torch.float32).to(DEVICE)
            cont_batch = torch.tensor(contours_arr[start:end], dtype=torch.float32).to(DEVICE)
            lang_batch = torch.full((end - start,), lang_id, dtype=torch.long, device=DEVICE)

            with torch.no_grad():
                out = model(wav_batch, feat_batch, cont_batch, lang_batch)
                probs = F.softmax(out['logits'], dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)

        #df['label'] = (np.array(all_probs) >= 0.5).astype(int)
        thresh = thresholds.get(lang, 0.5)
        df['label'] = (np.array(all_probs) >= thresh).astype(int)

        out_path = f"{config.output_path}/{lang}.csv"
        df.to_csv(out_path, index=False)
        print(f"  {lang:12s}: {len(df)} samples, {df['label'].mean()*100:.1f}% hate → {out_path}")

In [ ]:
def run_pipeline(config):
    print("=" * 70)
    print("PHAD-Net (Audio-Only)")
    print("=" * 70)

    prep = DataPreparator(config)
    prep.load_all_metadata()
    train_df, val_df = prep.create_splits()

    ext = FeatureExtractor(config)
    train_feats, train_cont = ext.extract_from_dataframe(train_df, "train")
    val_feats, val_cont = ext.extract_from_dataframe(val_df, "val")

    for name, (f, c) in [('train', (train_feats, train_cont)),
                          ('val', (val_feats, val_cont))]:
        f.to_csv(f"{config.output_path}/features/{name}.csv", index=False)
        np.save(f"{config.output_path}/features/{name}_contours.npy", c)

    train_loader, val_loader, scaler, cstats, feat_cols = create_data_loaders(
        train_feats, train_cont, val_feats, val_cont, config)

    model = PHADNet(
        num_handcrafted_features=len(feat_cols),
        num_languages=len(config.languages), cfg=config,
    ).to(DEVICE)

    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nParams: {n_total:,} total, {n_train:,} trainable ({n_train/n_total*100:.1f}%)")

    trainer = Trainer(model, config, DEVICE)
    model, history = trainer.train(train_loader, val_loader)
    with open(f"{config.output_path}/models/history.json", 'w') as f:
        json.dump(history, f, indent=2)

    print("\nComputing per-language thresholds on validation...")
    thresholds = compute_val_thresholds(model, val_loader, val_feats, DEVICE)

    run_inference(model, config, prep.label_encoder, scaler, cstats, feat_cols, thresholds)

    return model, history

In [27]:
if not os.path.exists(config.base_path):
    print(f"Dataset not found at {config.base_path}")
else:
    model, history = run_pipeline(config)

PHAD-Net (Audio-Only)
Loading metadata...


  0%|          | 0/10 [00:00<?, ?it/s]


Creating splits...
  Train: 7264, Val: 1448

Extracting features for train (7264 files)...


  0%|          | 0/7264 [00:00<?, ?it/s]

  Features: 84 dims, Contours: (7264, 4, 100)

Extracting features for val (1448 files)...


...

  tamil       : 77 samples, 42.9% hate → /workspace/outputs_phad_audio/tamil.csv


  telugu:   0%|          | 0/84 [00:00<?, ?it/s]

  telugu      : 84 samples, 79.8% hate → /workspace/outputs_phad_audio/telugu.csv


  spanish:   0%|          | 0/187 [00:00<?, ?it/s]

  spanish     : 187 samples, 84.0% hate → /workspace/outputs_phad_audio/spanish.csv
